In [ ]:
# Local models: prompt shoving ---> modeling ---> evaluate models 
# Imports
import os
import sys
from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI
from litellm import completion
# Add the utils folder to Python path
notebook_dir = os.path.dirname(os.path.abspath("__file__"))
pricer_path = os.path.join(notebook_dir, "../utils")
sys.path.insert(0, pricer_path)
# Class Item to validate the returned data and has lots of helper functions
from pricer.items import Item
from pricer.evaluator import evaluate

In [4]:
# Login to Hugging Face
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [5]:
# Download the batches dataset from Hugging Face 
username = "omarshaarawy11"
dataset = f"{username}/best_deal_03_batches"
train, val, test = Item.from_hub(dataset)
items = train + val + test
print(f"Loaded {len(items):,} items")

Loaded 21,091 items


In [6]:
# Invistigate the dataset
columns = ['title', 
'category', 
'price',
'full',
'weight',
'summary',
'prompt',
'id'
]
for i in columns:
    # Combine item with its attrs
    attr = getattr(items[0], i)
    print(attr)

Lurrose 100Pcs Full Cover Fake Toenails Artificial Transparent Nail Tips Nail Art for DIY
all beauty
6.99
None
0.0
Title: Lurrose 100pcs Full Cover Fake Toenails  
Category: Nail Art & Accessories  
Brand: Lurrose  
Description: High-quality, durable artificial toenails perfect for DIY nail art and manicures.  
Details: Features 100 lightweight, eco-friendly ABS nails that are easy to trim, shape, and fit comfortably to natural toenails, suitable for various styles and perfect as a gift.
None
None


In [7]:
# Prompt shoving
def prompt(item):
    # Don't put any constaints just generalize the respond
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": message}]

In [8]:
# Insvistigate summary feature of single datapoint
print(test[0].summary)

Title: Vzaahu Silicone Ice Cube Trays with Lid - 2 Pack  
Category: Kitchen & Dining  
Brand: Vzaahu  
Description: Durable, flexible silicone ice cube trays with lids for easy release and odor prevention.  
Details: Features food-grade silicone, stackable design, removable spill-resistant lid, dishwasher safe, and customizable with fruits or flavors.


In [9]:
prompt(test[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Vzaahu Silicone Ice Cube Trays with Lid - 2 Pack  \nCategory: Kitchen & Dining  \nBrand: Vzaahu  \nDescription: Durable, flexible silicone ice cube trays with lids for easy release and odor prevention.  \nDetails: Features food-grade silicone, stackable design, removable spill-resistant lid, dishwasher safe, and customizable with fruits or flavors.'}]

In [10]:
# Test dataset
def llama(item):
    response = completion(model="ollama/llama3.2", api_base="http://localhost:11434", messages=prompt(item))
    return response.choices[0].message.content

In [11]:
# Test single data point
llama(test[0])

'$15.99'

In [ ]:
# Evaluation
# We notice that base model performance is super bad so we need to fine-tune it with our data
evaluate(llama, test)

  0%|          | 0/2000 [00:00<?, ?it/s]

$7 $80 $0 $3 $2 $2 $6 $2 $5 $79 $15 $11 $20 $95 $1 $5 $32 $8 $270 $18 $41 $35 $80 $5 $5 $27 $0 $14 $6 $11 $40 $13 $275 $645 $17 $34 $9 $8 $44 $20 $2 $60 $3 $18 $15 $12 $70 $17 $2 $20 $4 $12 $7 $7 $10 $31 $5 $14 $4 $2 $15 $10 $23 $12 $29 $25 $9 $11 $30 $4 $3 $35 $22 $169 $85 $21 $20 $228 $5 $125 $10 $17 $2 $15 $75 $0 $60 $10 $4 $1 $0 $3 $20 $4 $36 $52 $4 $19 $8 $0 $15 $42 $58 $16 $110 $43 $2 $0 $17 $30 $33 $15 $45 $155 $40 $8 $23 $29 $18 $60 $83 $3 $1 $1 $125 $9 $1 $38 $17 $32 $20 $11 $597 $310 $0 $5 $10 $2 $24 $16 $13 $34 $4 $3 $1 $11 $15 $19 $7 $28 $20 $0 $16 $19 $80 $45 $0 $7 $130 $20 $30 $20 $38 $7 $68 $223 $23 $0 $70 $10 $5 $16 $724 $25 $1 $11 $5 $1 $17 $0 $20 $23 $6 $11 $158 $8 $55 $54 $14 $51 $37 $5 $5 $41 $23 $0 $72 $4 $23 $10 $10 $11 $15 $5 $11 $3 $20 $61 $14 $24 $110 $9 $28 $8 $14 $30 $35 $90 $0 $480 $2 $2 $18 $6 $15 $10 $82 $17 $12 $31 $34 $50 $36 $39 $6 $122 $105 $22 $25 $12 $14 $6 $14 $11 $18 $19 $1 $31 $30 $40 $24 $6 $19 $6 $95 $125 $90 $0 $45 $15 $14 $44 $27 $6 $7 $35 $2 